# **1. Perkenalan Dataset**


Tahap pertama, Anda harus mencari dan menggunakan dataset dengan ketentuan sebagai berikut:

1. **Sumber Dataset**:  
   * Dataset: Engine Fault Detection DataSource: Kaggle
   * Target: Engine_Condition (0=Normal, 1=Minor Fault, 2=Critical Fault)
   * Total Samples: 10.000
   * Features: 11 sensor readings

# **2. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# **3. Memuat Dataset**

Pada tahap ini, Anda perlu memuat dataset ke dalam notebook. Jika dataset dalam format CSV, Anda bisa menggunakan pustaka pandas untuk membacanya. Pastikan untuk mengecek beberapa baris awal dataset untuk memahami strukturnya dan memastikan data telah dimuat dengan benar.

Jika dataset berada di Google Drive, pastikan Anda menghubungkan Google Drive ke Colab terlebih dahulu. Setelah dataset berhasil dimuat, langkah berikutnya adalah memeriksa kesesuaian data dan siap untuk dianalisis lebih lanjut.

Jika dataset berupa unstructured data, silakan sesuaikan dengan format seperti kelas Machine Learning Pengembangan atau Machine Learning Terapan

In [ ]:
df = pd.read_csv('../engine_fault_raw/engine_fault_detection_dataset.csv')

print("Shape:", df.shape)
print("\nHead:")
df.head()

In [ ]:
print("Info:")
df.info()

In [ ]:
print("Missing values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())
print("\nClass distribution:\n", df['Engine_Condition'].value_counts())

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

In [ ]:
# 4.1 Statistik Deskriptif
df.describe()

In [ ]:
# 4.2 Distribusi Target
plt.figure(figsize=(6,4))
sns.countplot(x='Engine_Condition', data=df, palette='viridis')
plt.title('Distribusi Engine Condition')
plt.xticks([0,1,2], ['Normal','Minor Fault','Critical Fault'])
plt.tight_layout()
plt.savefig('class_distribution.png')
plt.show()

In [ ]:
# 4.3 Distribusi Fitur
df.drop('Engine_Condition', axis=1).hist(figsize=(14,10), bins=30, color='steelblue', edgecolor='black')
plt.suptitle('Distribusi Setiap Fitur', y=1.02)
plt.tight_layout()
plt.savefig('feature_distributions.png')
plt.show()

In [ ]:
# 4.4 Correlation Heatmap
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('correlation_heatmap.png')
plt.show()

In [ ]:
# 4.5 Boxplot per kelas
features = df.columns.drop('Engine_Condition')
fig, axes = plt.subplots(4, 3, figsize=(16, 12))
axes = axes.flatten()
for i, col in enumerate(features):
    sns.boxplot(x='Engine_Condition', y=col, data=df, ax=axes[i], palette='Set2')
    axes[i].set_title(col)
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.savefig('boxplot_per_class.png')
plt.show()

# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Jika Anda menggunakan data teks, data mentah sering kali mengandung nilai kosong, duplikasi, atau rentang nilai yang tidak konsisten, yang dapat memengaruhi kinerja model. Oleh karena itu, proses ini bertujuan untuk membersihkan dan mempersiapkan data agar analisis berjalan optimal.

Berikut adalah tahapan-tahapan yang bisa dilakukan, tetapi **tidak terbatas** pada:
1. Menghapus atau Menangani Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Deteksi dan Penanganan Outlier
5. Encoding Data Kategorikal
6. Binning (Pengelompokan Data)

Cukup sesuaikan dengan karakteristik data yang kamu gunakan yah. Khususnya ketika kami menggunakan data tidak terstruktur.

In [ ]:
# 5.1 Hapus duplikat
df = df.drop_duplicates()
print("Shape setelah drop duplicate:", df.shape)

In [ ]:
# 5.2 Handle Missing Values
df = df.dropna()
print("Shape setelah dropna:", df.shape)

In [ ]:
# 5.3 Deteksi & Handle Outlier (IQR method) - hanya fitur numerik
def remove_outliers_iqr(df: pd.DataFrame, exclude_cols: list) -> pd.DataFrame:
    """
    FIX: IQR hanya pada class mayoritas (Normal=0),
    agar data minority (Minor Fault, Critical Fault) tidak ikut terbuang.
    """
    before = len(df)
    cols = [c for c in df.columns if c not in exclude_cols]
    target_col = exclude_cols[0]

    # Hitung batas IQR dari class Normal saja
    majority = df[df[target_col] == 0]
    bounds = {}
    for col in cols:
        Q1 = majority[col].quantile(0.25)
        Q3 = majority[col].quantile(0.75)
        IQR = Q3 - Q1
        bounds[col] = (Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

    # Filter hanya pada class Normal
    mask_normal = df[target_col] == 0
    for col in cols:
        low, high = bounds[col]
        mask_normal = mask_normal & (df[col] >= low) & (df[col] <= high)

    # Gabungkan: normal yang sudah bersih + semua minority tetap utuh
    df_clean = pd.concat([
        df[mask_normal],
        df[df[target_col] != 0]
    ]).reset_index(drop=True)

    print(f"[outlier] Removed {before - len(df_clean)} outlier rows (majority class only)")
    return df_clean

feature_cols = df.columns.drop('Engine_Condition').tolist()
df = remove_outliers_iqr(df, feature_cols)
print("Shape setelah remove outlier:", df.shape)

In [ ]:
# 5.4 Pisah Fitur & Target
X = df.drop('Engine_Condition', axis=1)
y = df['Engine_Condition']

In [ ]:
# 5.5 Normalisasi dengan StandardScaler
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [ ]:
# 5.6 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

In [ ]:
# 5.6b Apply SMOTE (hanya pada train)
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)
print("After SMOTE:", pd.Series(y_train).value_counts().sort_index())

In [ ]:
# 5.7 Simpan hasil preprocessing
os.makedirs('/content/preprocessing/engine_fault_preprocessing', exist_ok=True)

train_df = X_train.copy()
train_df['Engine_Condition'] = y_train.values
test_df = X_test.copy()
test_df['Engine_Condition'] = y_test.values

train_df.to_csv('engine_fault_preprocessing/train.csv', index=False)
test_df.to_csv('engine_fault_preprocessing/test.csv', index=False)

print("Preprocessing selesai. File disimpan.")